In [ ]:
! pip install -U accelerate
! pip install -U transformers

In [ ]:
import pandas as pd
from google.colab import drive

from sklearn.model_selection import train_test_split

from transformers import Trainer, TrainingArguments
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import BertTokenizer, BertForSequenceClassification

import torch
from torch.utils.data import Dataset, DataLoader

import os

In [ ]:
# # Bert
# # Initialize the tokenizer
# tokenizer = BertTokenizer.from_pretrained('wietsedv/bert-base-dutch-cased')
# # Load pre-trained model with a classification head
# model = BertForSequenceClassification.from_pretrained('wietsedv/bert-base-dutch-cased', num_labels=2)
# model_save_path = '/content/drive/MyDrive/eColab/GenCareAI/models/BertClassification'

# Roberta
# Initialize the tokenizer
tokenizer = RobertaTokenizer.from_pretrained('pdelobelle/robbert-v2-dutch-base')
# Load pre-trained model with a classification head
model = RobertaForSequenceClassification.from_pretrained('pdelobelle/robbert-v2-dutch-base', num_labels=2)
model_save_path = '/content/drive/MyDrive/eColab/GenCareAI/models/RobertaClassification'

os.makedirs(model_save_path, exist_ok=True)

In [ ]:
drive.mount('/content/drive')
df_train = pd.read_csv('/content/drive/MyDrive/eColab/GenCareAI/data/df_train.csv')
df_valid = pd.read_csv('/content/drive/MyDrive/eColab/GenCareAI/data/df_valid.csv')
df_test = pd.read_csv('/content/drive/MyDrive/eColab/GenCareAI/data/df_test.csv')

In [ ]:
train_texts, train_labels = df_train['rapportage'].tolist(), df_train['onrust'].astype(int).tolist()
valid_texts, valid_labels = df_valid['rapportage'].tolist(), df_valid['onrust'].astype(int).tolist()
test_texts, test_labels = df_test['rapportage'].tolist(), df_test['onrust'].astype(int).tolist()

In [ ]:
class AgitationDataset(Dataset):
    def __init__(self, texts, labels):
        # No need to convert to list again
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=512)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
train_dataset = AgitationDataset(train_texts, train_labels)
test_dataset = AgitationDataset(valid_texts, valid_labels)

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=test_dataset            # evaluation dataset
)

In [ ]:
# Train the model
trainer.train()

In [ ]:
model.save_pretrained(model_save_path)